In [1]:
import os
print("active dir", os.getcwd())
print("files", os.listdir("."))

active dir /data/Projects/P0
files ['.ipynb_checkpoints', 'shell.nix', 'P0-GPT.ipynb', 'data']


In [2]:
with open("data/poems.txt", "r", encoding="utf-8") as f:
  text_trash = f.read()

In [3]:
# delete Caps and rare simbols
cnt_upper = lambda x: x.lower() if sum(1 for ch in x if ch.isupper()) > 1 else x

TRANS = str.maketrans({'ъ': 'ь', 'ё': 'е', 'Ё': 'Е', '"': '', ";": ",",
                       "\u00A0": " ", '«': '', '»': '', '(': '', ')': '',
                       '|': '\n'})

proc_line = lambda line: ' '.join(map(cnt_upper, line.split()))
proc_text = lambda text: '\n'.join(map(proc_line, text.split('\n')))

text= '\n'.join(proc_text(text_trash).translate(TRANS).split('---'))

In [4]:
print(text[1008000:1009000])

ым, но, знать, грамота далася ему не от господа бога.
П а т р и а р х
Уж эти мне грамотеи! что еще выдумал! буду царем на Москве! Ах он, сосуд диавольский! Однако нечего царю и докладывать об этом, что тревожить отца-государя? Довольно будет обьявить о побеге дьяку Смирнову али дьяку Ефимьеву, эдака ересь! буду царем на Москве!.. Поймать, поймать врагоугодника, да и сослать в Соловецкий на вечное покаяние. Ведь это ересь, отец игумен.
И г у м е н
Ересь, святый владыко, сущая ересь.
царские палаты
Д в а с т о л ь н и к а.
П е р в ы й
Где государь?
В т о р о й
В своей опочивальне
Он заперся с каким-то колдуном.
П е р в ы й
Так, вот его любимая беседа:
Кудесники, гадатели, колдуньи. —
Все ворожит, что красная невеста.
Желал бы знать, о чем гадает он?
В т о р о й
Вот он идет. Угодно ли спросить?
П е р в ы й
Как он угрюм!
Уходят.
Ц а р ь
входит
Достиг я высшей власти,
Шестой уж год я царствую спокойно.
Но счастья нет моей душе. Не так ли
Мы смолоду влюбляемся и алчем
Утех любви, но только у

In [5]:
chars = sorted(list(set(text)))
voc_size = len(chars)
print(len(chars))
print(''.join(chars))

68

 !,-.:?АБВГДЕЖЗИКЛМНОПРСТУФХЦЧШЩЭЮЯабвгдежзийклмнопрстуфхцчшщыьэюя—


In [6]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda q: ''.join([itos[i] for i in q])

print((encode('саня лох')))

[53, 36, 49, 66, 1, 47, 50, 57]


In [7]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:1000])

tensor([54, 52, 50, 45, 46, 36,  0, 30, 54, 50,  1, 54, 62,  1, 42, 36, 40, 49,
        50,  1, 39, 47, 66, 40, 44, 60, 63,  1, 49, 36,  1, 40, 50, 52, 50, 39,
        55,  0, 10,  1, 53, 54, 50, 52, 50, 49, 41,  1, 50, 54,  1, 38, 41, 53,
        41, 47, 62, 57,  1, 51, 50, 40, 52, 55, 39,  7,  0, 15, 49, 36, 54, 63,
         3,  1, 43, 36, 37, 44, 47, 50,  1, 53, 41, 52, 40, 41, 59, 46, 50,  1,
        54, 52, 41, 38, 50, 39, 55,  1, 67,  0, 10, 53, 41,  1, 47, 44, 58, 50,
         1, 54, 38, 50, 41,  1, 38, 53, 51, 62, 57, 49, 55, 47, 50,  1, 38, 40,
        52, 55, 39,  5,  0, 16,  1, 43, 36, 59, 41, 48,  1, 54, 62,  1, 37, 41,
        42, 44, 60, 63,  1, 54, 50, 52, 50, 51, 47, 44, 38, 50,  0, 15, 36,  1,
        51, 52, 50, 48, 59, 36, 38, 60, 41, 45, 53, 66,  1, 54, 52, 50, 45, 46,
        50, 45,  1, 38, 50, 53, 47, 41, 40,  7,  5,  5,  0, 20, 36,  1, 54, 41,
        37, 66,  3,  1, 51, 50, 40, 37, 50, 59, 41, 49, 66, 53, 63,  1, 46, 52,
        36, 53, 44, 38, 50,  3,  0, 15, 

In [8]:
n = int(0.9*len(text))
train_data = data[:n]
val_data = data[n:]

In [9]:
torch.manual_seed(239)

In [10]:
block_size = 8
batch_size = 4

def make_batch(data, block_size, batch_size):
    def sample_indices():
        return torch.randint(len(data) - block_size, (batch_size,))

    def slice_data(indices):
        x = torch.stack(list(map(lambda i: data[i:i+block_size], indices.tolist())))
        y = torch.stack(list(map(lambda i: data[i+1:i+block_size+1], indices.tolist())))
        return x, y

    return lambda: slice_data(sample_indices())

train_batch = make_batch(train_data, block_size, batch_size)
val_batch = make_batch(val_data, block_size, batch_size)

x, y = train_batch()
print(x)
print(y)

tensor([[50, 48,  1, 53, 63, 41, 57, 36],
        [50, 39, 50,  1, 37, 50, 52, 36],
        [49, 55,  1,  8, 49, 49, 55,  1],
        [53, 54, 62, 52, 44,  2,  5,  5]])
tensor([[48,  1, 53, 63, 41, 57, 36, 47],
        [39, 50,  1, 37, 50, 52, 36,  3],
        [55,  1,  8, 49, 49, 55,  1, 38],
        [54, 62, 52, 44,  2,  5,  5,  0]])


In [11]:
# bigram
import torch
from torch.nn import functional as F

torch.manual_seed(239)

def make_bigram_model(voc_size):
    embedding_weight = torch.randn(voc_size, voc_size, requires_grad=True)
    
    def logits_fn(idx):
        # (B, T) -> (B, T, C) logits
        return F.embedding(idx, embedding_weight)

    def loss_fn(logits, targets):
        # (logits + targets) -> loss
        B, T, C = logits.shape
        return F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        
    return logits_fn, loss_fn, embedding_weight

# create model
model_logits, model_loss, weights = make_bigram_model(voc_size)

# using
logits = model_logits(x)
loss = model_loss(logits, y) 

print(loss)

tensor(4.8493, grad_fn=<NllLossBackward0>)


In [12]:
print(voc_size)

68


In [13]:
# generate module
torch.manual_seed(239)

def generate_step(logits_fn, idx, count, temperature=1.0):
    sequence = idx
    for _ in range(count):
        logits = logits_fn(sequence)[:, -1, :] / temperature
        probs = F.softmax(logits, dim=1)
        idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
        sequence = torch.cat((sequence, idx_next), dim=1) # (B, T+1)
        yield sequence

count = 100
idx = torch.zeros((1,1), dtype=torch.long)

gen = generate_step(model_logits, idx, count)
for i, seq in enumerate(gen, 1):
    if i % 101 == 0:
        print(decode(seq[0].tolist()))
    final_seq = seq

print(decode(final_seq[0].tolist()))


яЮЯБРщцЯчТ,—!щж-Цкпиэч
ьгВА—И?юОлыДшсжжЗт:Дп ЗэГВЕ—ц оНлЛдКЧЕЮ:эПИбМэлсЦксЭ.ХээчфБЛжТоээТЦюБэюшспЯтЮ


In [14]:
# trainer

def train(optimizer, logits_fn, loss_fn, get_batch, count):
    for step in range(count):
        x, y = get_batch()
        
        logits = logits_fn(x)
        loss = loss_fn(logits, y)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        yield loss.item()

In [15]:
logits_fn, loss_fn, weights = make_bigram_model(voc_size)
optimizer = torch.optim.AdamW([weights], lr=1e-2)
print(loss)
block_size = 64
batch_size = 32

train_batch = make_batch(train_data, block_size, batch_size)
tr = train(optimizer, logits_fn, loss_fn, train_batch, count=10000)

for step, loss_value in enumerate(tr, 1):
    pass
print(step, loss_value)

tensor(4.8493, grad_fn=<NllLossBackward0>)
10000 2.6516823768615723


In [17]:
gen = generate_step(model_logits, idx, count=100)
for i, seq in enumerate(gen, 1):
    pass

print(decode(seq[0].tolist()))


НаЭпрЧоЮйФ!чТАБэПЕкФБэМтовябЯпВтф,д
ЦХТдИиэНшЛРвйШЖЯЯнЭ
Еп!х
йК——БНю,ВЭ?-ТЕХШЧН ГхСВонзИЯчТзСзрнЮИЗХ
